# Lab: Grover's Algorithm with Qiskit

In this lab we implement **Grover's algorithm**, a quantum algorithm for searching an unstructured space quadratically faster than any classical algorithm.

We are given a black-box ("oracle") function

$$f: \{0, 1\}^n \rightarrow \{0, 1\}$$

with the **promise** that there is exactly one marked input $m \in \{0, 1\}^n$ for which $f(m) = 1$, and $f(x) = 0$ for every other $x$. Our job is to find $m$.

> **Note:** the single-marked-item promise is specific to this notebook, which keeps the analysis and the iteration count as simple as possible. In general, Grover's algorithm also applies when there are $M$ marked inputs with $1 \le M \le N$. The optimal number of iterations is then $R \approx \frac{\pi}{4}\sqrt{N/M}$, so more marked inputs require **fewer** iterations.

There are $N = 2^n$ possible inputs.

- A classical algorithm needs up to $N$ oracle queries in the worst case.
- Grover's algorithm finds $m$ using only $O(\sqrt{N})$ oracle queries.

The algorithm works by repeatedly applying the **Grover operator**, which combines the oracle with a reflection about the uniform superposition:

$$G = D\,O, \qquad D = 2|s\rangle\langle s| - I, \qquad |s\rangle = H^{\otimes n}|0\rangle.$$

The oracle marks the solution with a phase, and the diffusion operator $D$ converts that phase into an amplitude that grows with each iteration.

We will build the algorithm in three tasks:
1. Implement the phase oracle $O$ for a given marked state.
2. Implement the diffusion operator $D$.
3. Implement the search algorithm that runs the circuit and returns the marked state.


In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit.circuit.library import ZGate
from qiskit_aer import AerSimulator

print("Qiskit imported successfully!")

Qiskit imported successfully!


### Helper Functions & Guardrails

The oracle acts on the $n$ input qubits and marks a single basis state with a phase. The helpers below are used by the test cells:

- `validate_marked(marked)` checks that `marked` is a non-empty binary string.
- `validate_oracle_inputs(n, oracle)` checks that `n` is valid and that the oracle acts on `n` qubits.
- `all_basis_states(n)` returns every $n$-bit input as a bitstring.
- `oracle_phase(oracle, n, x)` prepares $|x\rangle$, applies the oracle, and returns the sign of the amplitude of $|x\rangle$ (`+1` or `-1`).
- `grover_iterations(n)` returns the number of Grover iterations that maximizes the success probability for a single marked item.


In [2]:
def validate_marked(marked: str) -> None:
    """Guardrail: checks marked is a non-empty binary string."""
    assert isinstance(marked, str), "The marked state must be a string."
    assert len(marked) >= 1, "The marked state must have at least one bit."
    assert set(marked).issubset({'0', '1'}), (
        f"Marked state '{marked}' must only contain '0' and '1'."
    )


def validate_oracle_inputs(n: int, oracle: QuantumCircuit | None = None) -> None:
    """Guardrail: checks n is a positive integer and the oracle has n qubits."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    if oracle is not None:
        assert oracle.num_qubits == n, (
            f"Oracle must act on n = {n} qubits, but it acts on {oracle.num_qubits}."
        )


def all_basis_states(n: int) -> list[str]:
    """Returns every n-bit computational basis state as a bitstring."""
    return [format(i, f'0{n}b') for i in range(2 ** n)]


def oracle_phase(oracle: QuantumCircuit, n: int, x: str) -> int:
    """
    Returns the phase that the oracle assigns to the basis state |x>.

    Prepares |x>, applies the oracle, and reads the sign of the amplitude of
    |x>: +1 if the phase is positive, -1 if it is negative.
    """
    validate_oracle_inputs(n, oracle)
    assert len(x) == n, f"Input '{x}' must have length n = {n}."
    assert set(x).issubset({'0', '1'}), f"Input '{x}' must be a binary string."

    circuit = QuantumCircuit(n)
    for i, bit in enumerate(reversed(x)):
        if bit == '1':
            circuit.x(i)
    circuit.compose(oracle, inplace=True)

    statevector = Statevector(circuit)
    amplitude = complex(statevector.data[int(x, 2)])
    return 1 if amplitude.real > 0 else -1


def grover_iterations(n: int) -> int:
    """Returns the optimal number of Grover iterations for one marked item."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    return max(1, int(np.pi / 4 * np.sqrt(2 ** n)))


print("Helper functions and guardrails loaded successfully!")


Helper functions and guardrails loaded successfully!


## Task 1: The Phase Oracle

An oracle is a reversible circuit that marks the solution with a phase:

$$O |x\rangle = (-1)^{f(x)} |x\rangle.$$

That is, $O$ leaves every basis state unchanged except the marked state $|m\rangle$, whose amplitude is multiplied by $-1$.

A convenient way to build $O$ is:

1. Apply an $X$ gate to every qubit where `marked` has a `0`. This maps $|m\rangle$ to $|1\cdots 1\rangle$.
2. Apply a multi-controlled $Z$ gate on all $n$ qubits. This flips the sign of $|1\cdots 1\rangle$ and nothing else.
3. Undo step 1 by applying the same $X$ gates again.

Because the $X$ gates cancel, every other basis state is unaffected, while the marked state picks up a phase of $-1$.


### Task 1.1: Implement the Oracle

Implement `grover_oracle(circuit: QuantumCircuit, marked: str) -> QuantumCircuit`.

- `circuit`: an `n`-qubit circuit, where $n = $ `len(marked)`.
- `marked`: the marked bitstring of length $n$.
- Returns the same circuit with the oracle gates applied.

> Hint: Qiskit uses little-endian ordering, so `marked[-1-i]` corresponds to qubit `i`. A multi-controlled $Z$ over all `n` qubits is `ZGate().control(n - 1)` applied to `range(n)` (this also works for $n = 1$).


In [ ]:
def grover_oracle(circuit: QuantumCircuit, marked: str) -> QuantumCircuit:
    """
    Applies the phase oracle that marks |marked> to the given circuit.

    Args:
        circuit: An n-qubit circuit, where n = len(marked).
        marked: The marked bitstring of length n.

    Returns:
        QuantumCircuit: The same circuit with the oracle gates applied.
    """
    validate_marked(marked)
    validate_oracle_inputs(len(marked), circuit)
    
    n = len(marked)

    for i in range(n):
        if marked[-1-i] == '0':
            circuit.x(i)

    controlled_z = ZGate().control(n - 1)

    qubits = []
    for i in range(n):
        qubits.append(i)

    circuit.append(controlled_z, qubits)

    for i in range(n):
        if marked[-1-i] == '0':
            circuit.x(i)

    return circuit
    # TODO: apply the oracle gates to the given circuit
    pass


### Task 1.2: Test the Oracle

The cell below checks that your oracle flips the sign of the marked state and leaves every other basis state untouched, for all marked states on 1, 2, and 3 qubits.


In [8]:
for n in [1, 2, 3]:
    for marked in all_basis_states(n):
        oracle = grover_oracle(QuantumCircuit(n), marked)
        for x in all_basis_states(n):
            expected = -1 if x == marked else 1
            observed = oracle_phase(oracle, n, x)
            assert observed == expected, (
                f"Oracle for marked={marked} gave phase {observed} on x={x}, "
                f"expected {expected}"
            )

print("Oracle passed all tests!")


Oracle passed all tests!


## Task 2: The Diffusion Operator

The diffusion operator reflects the state about the uniform superposition $|s\rangle = H^{\otimes n}|0\rangle$:

$$D = 2|s\rangle\langle s| - I.$$

Conjugating by Hadamards turns this into a reflection about $|0\cdots 0\rangle$:

$$D = H^{\otimes n}\left(2|0\rangle\langle 0| - I\right)H^{\otimes n}.$$

The inner reflection is easy to build: an $X$ on every qubit sends $|0\cdots 0\rangle$ to $|1\cdots 1\rangle$, a multi-controlled $Z$ flips the sign of that one state, and the $X$ gates undo the change. Altogether,

$$D = H^{\otimes n}\, X^{\otimes n}\, \text{MCZ}\, X^{\otimes n}\, H^{\otimes n}.$$


### Task 2.1: Implement the Diffusion Operator

Implement `grover_diffusion(n: int) -> QuantumCircuit`.

- `n`: number of qubits.
- Returns an `n`-qubit circuit implementing $D$ (with no measurements).

> Hint: build the circuit as $H^{\otimes n}$, then $X^{\otimes n}$, then `ZGate().control(n - 1)` on all qubits, then $X^{\otimes n}$, then $H^{\otimes n}$.


In [ ]:
def grover_diffusion(n: int) -> QuantumCircuit:
    """
    Builds the n-qubit diffusion operator D = 2|s><s| - I.

    Args:
        n: Number of qubits.

    Returns:
        QuantumCircuit: An n-qubit circuit implementing the diffusion operator.
    """
    validate_oracle_inputs(n)

    # TODO: build and return the diffusion operator
    pass


### Task 2.2: Test the Diffusion Operator

The cell below checks that `grover_diffusion(n)` is a unitary matrix that equals its own inverse and leaves the uniform superposition $|s\rangle$ unchanged.


In [ ]:
for n in [1, 2, 3, 4]:
    diffusion = grover_diffusion(n)
    operator = Operator(diffusion)
    assert operator.is_unitary(), f"Diffusion for n={n} is not unitary."

    # D is a reflection, so it is its own inverse: D^2 = I.
    assert (operator @ operator).equiv(Operator(np.eye(2 ** n))), (
        f"Diffusion for n={n} does not square to the identity."
    )

    # D leaves the uniform superposition |s> unchanged.
    prep = QuantumCircuit(n)
    prep.h(range(n))
    uniform = Statevector(prep)
    test = prep.copy()
    test.compose(diffusion, inplace=True)
    assert Statevector(test).equiv(uniform), (
        f"Diffusion for n={n} does not preserve the uniform superposition."
    )

print("Diffusion operator passed all tests!")


## Task 3: The Grover Search Algorithm

The full algorithm alternates the oracle with the diffusion operator, $R$ times:

```
     ┌───┐ ┌─────────────────────────┐
q_0: ┤ H ├─┤                         ├─ M
q_1: ┤ H ├─┤   Grover operator (×R)  ├─ M
  ⋮    ⋮   │       G = D · O         │  ⋮
q_n: ┤ H ├─┤                         ├─ M
     └───┘ └─────────────────────────┘
```

After $R$ iterations the amplitude of the marked state is $\sin((2R+1)\theta)$, where $\sin\theta = 1/\sqrt{N}$. Choosing

$$R = \left\lfloor \frac{\pi}{4}\sqrt{N} \right\rfloor$$

makes the marked state the most likely measurement outcome; use the provided `grover_iterations(n)` helper. (As in the introduction, this formula assumes the single marked item of this notebook; with $M$ marked items it becomes $R = \left\lfloor \frac{\pi}{4}\sqrt{N/M} \right\rfloor$.)

Implement `grover_search(n: int, oracle: QuantumCircuit) -> str`.

- `n`: number of input qubits.
- `oracle`: an `n`-qubit phase oracle marking a single basis state.
- Returns the marked bitstring of length `n`.

> Hint: build an `n`-qubit, `n`-clbit circuit, apply $H^{\otimes n}$, then repeat `grover_iterations(n)` times: compose the oracle and then `grover_diffusion(n)`. Measure every qubit and run the circuit on `AerSimulator`. Run the circuit through `transpile(circuit, simulator)` first, because the simulator does not natively support large multi-controlled gates. Return the most frequent measured bitstring.


In [ ]:
def grover_search(n: int, oracle: QuantumCircuit) -> str:
    """
    Runs Grover's algorithm and returns the marked bitstring.

    Args:
        n: Number of input qubits.
        oracle: An n-qubit phase oracle marking a single basis state.

    Returns:
        str: The measured marked bitstring of length n.
    """
    validate_oracle_inputs(n, oracle)

    # TODO: build the Grover circuit, run it on AerSimulator, and return the
    # most frequent measured bitstring.
    pass


### Final Test

The cell below searches for every marked state on 2, 3, 4, and 5 qubits using your implementation. It must pass for the lab to be complete.


In [ ]:
for n in [2, 3, 4, 5]:
    for marked in all_basis_states(n):
        oracle = grover_oracle(QuantumCircuit(n), marked)
        recovered = grover_search(n, oracle)
        assert recovered == marked, (
            f"Failed to find marked state n={n}, marked={marked}: got {recovered}"
        )

print("Grover's algorithm passed all tests!")
